In [1]:
!pip install optuna
!pip install fasttext


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 2.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-3.0.4-py3-none-any.whl.metadata (10 kB)
Using cached pybind11-3.0.4-py3-none-any.whl (314 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp312-cp312-linux_x86_64.whl size=4653910 sha256=fe58fa5b873c128371260a93806738c89c993d993bf61788fa4b57574b2d93a7
  Stored in directory: /root/.cache/pip/wheels/20/27/95/a7baf1b435f1cbde017cabdf1e9688526d2b0e929255a359c6
Successfully built fasttext


In [2]:
# Importing libraries

import os
import re
import pandas as pd
import numpy as np
import fasttext
import optuna

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix

)

In [4]:
# Upload cleaned dataset

from google.colab import files

uploaded = files.upload()

Saving df_filtered_cleaned.csv to df_filtered_cleaned.csv


In [5]:
# Load cleaned dataset

df_filtered = pd.read_csv("df_filtered_cleaned.csv")

print(df_filtered.shape)
df_filtered.head()

(32030, 3)


,author_ID,post_cleaned,political_leaning
0,t2_7ramzeng,you can buy the show and stream it through the...,right
1,t2_7ramzeng,me want to play q bert holy shit based alex jo...,right
2,t2_7ramzeng,shouldn t rely on any external services or per...,right
3,t2_7ramzeng,pr to a specific person usually that just mean...,right
4,t2_7ramzeng,this article s intention is clear that they wa...,right


In [6]:
# Author level train test split

unique_authors = df_filtered["author_ID"].unique()

train_authors, test_authors = train_test_split(
    unique_authors,
    test_size=0.20,
    random_state=42
)

train_df = df_filtered[df_filtered["author_ID"].isin(train_authors)].copy()
test_df = df_filtered[df_filtered["author_ID"].isin(test_authors)].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

overlap_authors = set(train_df["author_ID"]).intersection(set(test_df["author_ID"]))
print("Overlapping authors:", len(overlap_authors))

print("\nTrain class distribution:")
print(train_df["political_leaning"].value_counts(normalize=True))

print("\nTest class distribution:")
print(test_df["political_leaning"].value_counts(normalize=True))

Train shape: (26606, 3)
Test shape: (5424, 3)
Overlapping authors: 0

Train class distribution:
political_leaning
right    0.548185
left     0.451815
Name: proportion, dtype: float64

Test class distribution:
political_leaning
right    0.528945
left     0.471055
Name: proportion, dtype: float64


In [7]:
# Internal author level train validation split for Optuna

train_unique_authors = train_df["author_ID"].unique()

optuna_train_authors, optuna_val_authors = train_test_split(
    train_unique_authors,
    test_size=0.20,
    random_state=42
)

optuna_train_df = train_df[train_df["author_ID"].isin(optuna_train_authors)].copy()
optuna_val_df = train_df[train_df["author_ID"].isin(optuna_val_authors)].copy()

print("Optuna train shape:", optuna_train_df.shape)
print("Optuna validation shape:", optuna_val_df.shape)

overlap_optuna_authors = set(optuna_train_df["author_ID"]).intersection(
    set(optuna_val_df["author_ID"])
)

print("Overlapping authors between Optuna train and validation:", len(overlap_optuna_authors))

print("\nOptuna train class distribution:")
print(optuna_train_df["political_leaning"].value_counts(normalize=True))

print("\nOptuna validation class distribution:")
print(optuna_val_df["political_leaning"].value_counts(normalize=True))

Optuna train shape: (22137, 3)
Optuna validation shape: (4469, 3)
Overlapping authors between Optuna train and validation: 0

Optuna train class distribution:
political_leaning
right    0.550752
left     0.449248
Name: proportion, dtype: float64

Optuna validation class distribution:
political_leaning
right    0.535467
left     0.464533
Name: proportion, dtype: float64


In [8]:
# Helper functions for fastText

def clean_for_fasttext(text):
    text = str(text).replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def write_fasttext_file(df, text_col, label_col, output_path):
    temp_df = df.copy()
    temp_df["fasttext_line"] = (
        "__label__" + temp_df[label_col].astype(str) + " " +
        temp_df[text_col].apply(clean_for_fasttext)
    )

    temp_df[["fasttext_line"]].to_csv(
        output_path,
        index=False,
        header=False
    )


def predict_fasttext_labels(model, texts):
    predictions = []

    for text in texts:
        labels, probabilities = model.predict(clean_for_fasttext(text), k=1)
        predictions.append(labels[0].replace("__label__", ""))

    return predictions


def evaluate_predictions(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)

    precision, recall, macro_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    return accuracy, precision, recall, macro_f1

In [10]:
#fastText files for Optuna tuning

write_fasttext_file(
    optuna_train_df,
    text_col="post_cleaned",
    label_col="political_leaning",
    output_path="fasttext_optuna_train.txt"
)

write_fasttext_file(
    optuna_val_df,
    text_col="post_cleaned",
    label_col="political_leaning",
    output_path="fasttext_optuna_val.txt"
)

In [11]:
# Define Optuna objective function for fastText

def objective(trial):
    lr = trial.suggest_float("lr", 0.01, 0.15, log=True)
    epoch = trial.suggest_categorical("epoch", [10, 25, 50])
    wordNgrams = trial.suggest_categorical("wordNgrams", [1, 2, 3])
    dim = trial.suggest_categorical("dim", [50, 100])
    minCount = trial.suggest_categorical("minCount", [1, 3, 5])

    try:
        model = fasttext.train_supervised(
            input="fasttext_optuna_train.txt",
            lr=lr,
            epoch=epoch,
            wordNgrams=wordNgrams,
            dim=dim,
            minCount=minCount,
            loss="softmax",
            verbose=0
        )

        y_val_true = optuna_val_df["political_leaning"].tolist()
        y_val_pred = predict_fasttext_labels(
            model,
            optuna_val_df["post_cleaned"].tolist()
        )

        _, _, _, val_macro_f1 = evaluate_predictions(y_val_true, y_val_pred)

        return val_macro_f1

    except RuntimeError:
        return 0.0

In [13]:
# Patch fastText predict function for NumPy 2 compatibility

import fasttext.FastText as ft_module
import numpy as np

def patched_predict(self, text, k=1, threshold=0.0, on_unicode_error="strict"):
    if isinstance(text, list):
        all_labels = []
        all_probs = []

        for t in text:
            labels, probs = patched_predict(
                self,
                t,
                k=k,
                threshold=threshold,
                on_unicode_error=on_unicode_error
            )
            all_labels.append(labels)
            all_probs.append(probs)

        return all_labels, all_probs

    text = str(text).replace("\n", " ")
    predictions = self.f.predict(text, k, threshold, on_unicode_error)

    if predictions:
        probs, labels = zip(*predictions)
    else:
        probs, labels = ([], ())

    return labels, np.asarray(probs)

ft_module._FastText.predict = patched_predict

In [14]:
# Define Optuna objective function for fastText

val_texts = optuna_val_df["post_cleaned"].tolist()
y_val_true = optuna_val_df["political_leaning"].tolist()

def objective(trial):
    lr = trial.suggest_float("lr", 0.005, 0.08, log=True)
    epoch = trial.suggest_categorical("epoch", [10, 25])
    wordNgrams = trial.suggest_categorical("wordNgrams", [1, 2])

    try:
        model = fasttext.train_supervised(
            input="fasttext_optuna_train.txt",
            lr=lr,
            epoch=epoch,
            wordNgrams=wordNgrams,
            dim=50,
            minCount=1,
            loss="softmax",
            verbose=0
        )

        y_val_pred = predict_fasttext_labels(model, val_texts)

        _, _, _, val_macro_f1 = evaluate_predictions(
            y_val_true,
            y_val_pred
        )

        return val_macro_f1

    except Exception as e:
        print("Trial failed:", e)
        return 0.0

In [23]:
best_ft_params = {
    "lr": 0.07226,
    "epoch": 10,
    "wordNgrams": 1,
    "dim": 25,
    "minCount": 1
}

write_fasttext_file(
    train_df,
    text_col="post_cleaned",
    label_col="political_leaning",
    output_path="fasttext_train_full.txt"
)

write_fasttext_file(
    test_df,
    text_col="post_cleaned",
    label_col="political_leaning",
    output_path="fasttext_test.txt"
)

ft_model_optuna = fasttext.train_supervised(
    input="fasttext_train_full.txt",
    lr=best_ft_params["lr"],
    epoch=best_ft_params["epoch"],
    wordNgrams=best_ft_params["wordNgrams"],
    dim=50,
    minCount=1,
    loss="softmax",
    verbose=0
)

y_test_true_ft = test_df["political_leaning"].tolist()
y_test_pred_ft_optuna = predict_fasttext_labels(ft_model_optuna, test_df["post_cleaned"].tolist())

ft_optuna_accuracy, ft_optuna_precision, ft_optuna_recall, ft_optuna_f1 = evaluate_predictions(
    y_test_true_ft, y_test_pred_ft_optuna
)

In [15]:
# Run Optuna study for fastText

ft_study = optuna.create_study(direction="maximize")
ft_study.optimize(objective, n_trials=8)

print("Best validation Macro F1:", ft_study.best_value)
print("Best parameters:")
ft_study.best_params

[I 2026-05-21 19:37:08,416] A new study created in memory with name: no-name-4e658081-ca26-4c50-9fe2-b3401d18bb18
[I 2026-05-21 19:45:10,546] Trial 0 finished with value: 0.6281229811664537 and parameters: {'lr': 0.04245127704827752, 'epoch': 25, 'wordNgrams': 2}. Best is trial 0 with value: 0.6281229811664537.
[I 2026-05-21 19:48:39,619] Trial 1 finished with value: 0.4627463664043693 and parameters: {'lr': 0.03698701074814828, 'epoch': 10, 'wordNgrams': 2}. Best is trial 0 with value: 0.6281229811664537.
[I 2026-05-21 19:50:08,533] Trial 2 finished with value: 0.5752441753999851 and parameters: {'lr': 0.02356804354807287, 'epoch': 10, 'wordNgrams': 1}. Best is trial 0 with value: 0.6281229811664537.
[I 2026-05-21 19:58:55,158] Trial 3 finished with value: 0.42421860311908155 and parameters: {'lr': 0.014675483637309729, 'epoch': 25, 'wordNgrams': 2}. Best is trial 0 with value: 0.6281229811664537.
[I 2026-05-21 20:07:39,386] Trial 4 finished with value: 0.6218468659047176 and paramete

Best validation Macro F1: 0.6331694004405842
Best parameters:


{'lr': 0.05131739225699868, 'epoch': 25, 'wordNgrams': 2}

In [18]:
# Train final fastText model with Optuna selected parameters

best_ft_params = ft_study.best_params

write_fasttext_file(
    train_df,
    text_col="post_cleaned",
    label_col="political_leaning",
    output_path="fasttext_train_full.txt"
)

write_fasttext_file(
    test_df,
    text_col="post_cleaned",
    label_col="political_leaning",
    output_path="fasttext_test.txt"
)

ft_model_optuna = fasttext.train_supervised(
    input="fasttext_train_full.txt",
    lr=best_ft_params["lr"],
    epoch=best_ft_params["epoch"],
    wordNgrams=best_ft_params["wordNgrams"],
    dim=50,
    minCount=1,
    loss="softmax",
    verbose=0
)

y_test_true_ft = test_df["political_leaning"].tolist()
y_test_pred_ft_optuna = predict_fasttext_labels(
    ft_model_optuna,
    test_df["post_cleaned"].tolist()
)

ft_optuna_accuracy, ft_optuna_precision, ft_optuna_recall, ft_optuna_f1 = evaluate_predictions(
    y_test_true_ft,
    y_test_pred_ft_optuna
)

print("Optuna selected fastText test results")
print("Accuracy:", ft_optuna_accuracy)
print("Macro Precision:", ft_optuna_precision)
print("Macro Recall:", ft_optuna_recall)
print("Macro F1:", ft_optuna_f1)

print("\nClassification Report:")
print(classification_report(y_test_true_ft, y_test_pred_ft_optuna, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_true_ft, y_test_pred_ft_optuna, labels=["left", "right"]))

Optuna selected fastText test results
Accuracy: 0.6504424778761062
Macro Precision: 0.65062983919708
Macro Recall: 0.6511304115318688
Macro F1: 0.6502094409230748

Classification Report:
              precision    recall  f1-score   support

        left       0.62      0.66      0.64      2555
       right       0.68      0.64      0.66      2869

    accuracy                           0.65      5424
   macro avg       0.65      0.65      0.65      5424
weighted avg       0.65      0.65      0.65      5424


Confusion Matrix:
[[1694  861]
 [1035 1834]]


In [22]:
# Overfit for Optuna selected fastText

y_train_true_ft = train_df["political_leaning"].tolist()

y_train_pred_ft_optuna = predict_fasttext_labels(
    ft_model_optuna,
    train_df["post_cleaned"].tolist()
)

ft_optuna_train_accuracy, ft_optuna_train_precision, ft_optuna_train_recall, ft_optuna_train_f1 = evaluate_predictions(
    y_train_true_ft,
    y_train_pred_ft_optuna
)

fasttext_optuna_diagnostic = pd.DataFrame({
    "model": ["fastText Optuna selected"],
    "train_accuracy": [ft_optuna_train_accuracy],
    "test_accuracy": [ft_optuna_accuracy],
    "train_macro_f1": [ft_optuna_train_f1],
    "test_macro_f1": [ft_optuna_f1],
    "accuracy_gap": [ft_optuna_train_accuracy - ft_optuna_accuracy],
    "macro_f1_gap": [ft_optuna_train_f1 - ft_optuna_f1],
    "lr": [best_ft_params["lr"]],
    "epoch": [best_ft_params["epoch"]],
    "wordNgrams": [best_ft_params["wordNgrams"]],
    "dim": [50],
    "minCount": [1]
})

fasttext_optuna_diagnostic

,model,train_accuracy,test_accuracy,train_macro_f1,test_macro_f1,accuracy_gap,macro_f1_gap,lr,epoch,wordNgrams,dim,minCount
0,fastText Optuna selected,0.706307,0.650442,0.705665,0.650209,0.055864,0.055456,0.07226,10,1,50,1


In [24]:
print("Test Macro F1:", ft_optuna_f1)
print("Test Accuracy:", ft_optuna_accuracy)

print(classification_report(y_test_true_ft, y_test_pred_ft_optuna, zero_division=0))

Test Macro F1: 0.6360357689904635
Test Accuracy: 0.6360619469026548
              precision    recall  f1-score   support

        left       0.60      0.68      0.64      2555
       right       0.68      0.59      0.63      2869

    accuracy                           0.64      5424
   macro avg       0.64      0.64      0.64      5424
weighted avg       0.64      0.64      0.64      5424



In [25]:
y_train_true_ft = train_df["political_leaning"].tolist()
y_train_pred_ft = predict_fasttext_labels(ft_model_optuna, train_df["post_cleaned"].tolist())

_, _, _, train_f1 = evaluate_predictions(y_train_true_ft, y_train_pred_ft)

print("Train Macro F1:", round(train_f1, 4))
print("Test Macro F1:", round(ft_optuna_f1, 4))
print("Gap:", round(train_f1 - ft_optuna_f1, 4))

Train Macro F1: 0.7187
Test Macro F1: 0.636
Gap: 0.0826


# SHAP/LIME ABLATION

In [30]:
!pip install lime -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [26]:
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

eu_keywords = pd.read_csv("eu-political-barometer_keywords_2024_v01.csv")
eu_words = set(eu_keywords["word"].astype(str).str.lower().str.strip())

harvard_lexicon = pd.read_csv("lexicon.csv", header=None, names=["word"])
harvard_words = set(harvard_lexicon["word"].astype(str).str.lower().str.strip())

political_lexicon = eu_words.union(harvard_words)
political_lexicon = {w for w in political_lexicon if w not in ENGLISH_STOP_WORDS and len(w) > 2}

print("Combined lexicon size:", len(political_lexicon))

Combined lexicon size: 19998


In [27]:
# LIME
def predict_proba_ft(texts):
    results = []
    for text in texts:
        text = str(text).replace("\n", " ")
        labels, probs = ft_model_optuna.predict(text, k=2)
        label_prob = {l.replace("__label__", ""): p for l, p in zip(labels, probs)}
        left_prob = label_prob.get("left", 0.0)
        right_prob = label_prob.get("right", 0.0)
        results.append([left_prob, right_prob])
    return np.array(results)

In [32]:
from lime.lime_text import LimeTextExplainer
from collections import defaultdict

class_names = ["left", "right"]
lime_explainer_ft = LimeTextExplainer(class_names=class_names)

lime_feature_scores_ft = defaultdict(list)
lime_sample_size = 20

np.random.seed(42)
sample_indices = np.random.choice(len(test_df), size=lime_sample_size, replace=False)

print("Running LIME for fastText...")
for i, idx in enumerate(sample_indices):
    text = test_df["post_cleaned"].iloc[idx]

    exp = lime_explainer_ft.explain_instance(
        text,
        predict_proba_ft,
        num_features=20,
        num_samples=500,
        labels=[0, 1]
    )

    for label_idx in [0, 1]:
        for feature, score in exp.as_list(label=label_idx):
            feature = str(feature).lower().strip()
            lime_feature_scores_ft[feature].append(score if label_idx == 1 else -score)

    if (i + 1) % 20 == 0:
        print(f"{i+1}/{lime_sample_size} done")

Running LIME for fastText...
20/20 done


In [33]:
lime_global_ft = pd.DataFrame([
    {
        "feature": feature,
        "mean_abs_lime": np.abs(scores).mean(),
        "mean_lime": np.mean(scores),
        "frequency": len(scores)
    }
    for feature, scores in lime_feature_scores_ft.items()
])

lime_global_ft = lime_global_ft[lime_global_ft["frequency"] >= 2].copy()
lime_global_ft["direction"] = np.where(lime_global_ft["mean_lime"] > 0, "right", "left")
lime_global_ft = lime_global_ft.sort_values("mean_abs_lime", ascending=False).reset_index(drop=True)

lime_top_200_ft = lime_global_ft.head(200)
lime_top_200_ft.head(10)

,feature,mean_abs_lime,mean_lime,frequency,direction
0,that,0.217709,-0.217709,36,left
1,is,0.150683,0.150683,32,right
2,yeah,0.141434,-0.141434,6,left
3,t,0.124478,0.124478,32,right
4,they,0.120791,0.120791,8,right
5,government,0.116651,0.116651,6,right
6,game,0.111163,0.111163,2,right
7,and,0.102804,-0.102804,36,left
8,this,0.099614,-0.099614,20,left
9,never,0.093822,0.093822,2,right


In [34]:
!pip install shap -q
import shap

def predict_for_shap_ft(texts):
    return predict_proba_ft(list(texts))

masker = shap.maskers.Text(tokenizer=r"\W+")
explainer_ft = shap.Explainer(predict_for_shap_ft, masker, output_names=["left", "right"])

shap_sample_ft = test_df["post_cleaned"].iloc[:50].tolist()

print("Running SHAP for fastText...")
shap_values_ft = explainer_ft(shap_sample_ft)

Running SHAP for fastText...


PartitionExplainer explainer: 51it [02:52,  3.52s/it]


In [35]:
shap_tokens_ft = []
shap_scores_left_ft = []
shap_scores_right_ft = []

for i in range(len(shap_values_ft)):
    tokens = shap_values_ft[i].data
    values = shap_values_ft[i].values

    for token, value in zip(tokens, values):
        token = str(token).lower().strip()
        if len(token) > 2:
            shap_tokens_ft.append(token)
            shap_scores_left_ft.append(value[0])
            shap_scores_right_ft.append(value[1])

shap_df_ft = pd.DataFrame({
    "token": shap_tokens_ft,
    "shap_left": shap_scores_left_ft,
    "shap_right": shap_scores_right_ft
})

shap_agg_ft = shap_df_ft.groupby("token").agg(
    mean_shap_left=("shap_left", "mean"),
    mean_shap_right=("shap_right", "mean"),
    mean_abs_shap=("shap_right", lambda x: np.abs(x).mean())
).reset_index()

shap_agg_ft["direction"] = np.where(
    shap_agg_ft["mean_shap_right"] > shap_agg_ft["mean_shap_left"],
    "right", "left"
)

shap_top_200_ft = shap_agg_ft.sort_values("mean_abs_shap", ascending=False).head(200)
shap_top_200_ft.head(10)

,token,mean_shap_left,mean_shap_right,mean_abs_shap,direction
4744,mentality,0.012334,-0.010380,0.010380,left
283,angelica,-0.005651,0.007604,0.007604,right
5213,nyt,0.008785,-0.006832,0.006832,left
5506,pause,-0.000756,0.005964,0.005964,right
5221,obligation,-0.004715,0.005692,0.005692,right
6447,ripped,-0.003702,0.005655,0.005655,right
1863,deceiving,-0.003683,0.005636,0.005636,right
6372,restrictions,-0.004623,0.005600,0.005600,right
1776,cups,0.007502,-0.005549,0.005549,left
4496,longtime,-0.003434,0.005388,0.005388,right


In [36]:
def feature_overlaps_lexicon(feature, lexicon):
    feature = str(feature).lower().strip()
    if feature in lexicon:
        return True
    tokens = feature.split()
    return any(token in lexicon for token in tokens)

shap_top_200_ft["lexicon_overlap"] = shap_top_200_ft["token"].apply(
    lambda x: feature_overlaps_lexicon(x, political_lexicon)
)

lime_top_200_ft["lexicon_overlap"] = lime_top_200_ft["feature"].apply(
    lambda x: feature_overlaps_lexicon(x, political_lexicon)
)

shap_overlap_ft = shap_top_200_ft[shap_top_200_ft["lexicon_overlap"]].copy()
lime_overlap_ft = lime_top_200_ft[lime_top_200_ft["lexicon_overlap"]].copy()

combined_ft = set(shap_top_200_ft["token"]).union(set(lime_top_200_ft["feature"]))
combined_ft_df = pd.DataFrame({"feature": sorted(combined_ft)})
combined_ft_df["lexicon_overlap"] = combined_ft_df["feature"].apply(
    lambda x: feature_overlaps_lexicon(x, political_lexicon)
)
combined_overlap_ft = combined_ft_df[combined_ft_df["lexicon_overlap"]].copy()

overlap_summary_ft = pd.DataFrame({
    "source": ["SHAP", "LIME", "Combined"],
    "total_features": [len(shap_top_200_ft), len(lime_top_200_ft), len(combined_ft_df)],
    "lexicon_overlap_count": [len(shap_overlap_ft), len(lime_overlap_ft), len(combined_overlap_ft)],
    "overlap_ratio": [
        round(len(shap_overlap_ft)/len(shap_top_200_ft), 3),
        round(len(lime_overlap_ft)/len(lime_top_200_ft), 3),
        round(len(combined_overlap_ft)/len(combined_ft_df), 3)
    ]
})

overlap_summary_ft

/tmp/ipykernel_2778/2526651732.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lime_top_200_ft["lexicon_overlap"] = lime_top_200_ft["feature"].apply(


,source,total_features,lexicon_overlap_count,overlap_ratio
0,SHAP,200,49,0.245
1,LIME,200,61,0.305
2,Combined,400,110,0.275


In [37]:
import re

ablation_terms_ft = combined_overlap_ft["feature"].astype(str).str.lower().tolist()

def remove_terms(text, terms):
    text = str(text).lower()
    terms_sorted = sorted(terms, key=len, reverse=True)
    for term in terms_sorted:
        pattern = r"\b" + re.escape(term) + r"\b"
        text = re.sub(pattern, " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

print("Ablation terms:", len(ablation_terms_ft))

Ablation terms: 110


In [38]:
# Create ablated train and test sets
train_df_ablated = train_df.copy()
train_df_ablated["post_cleaned"] = train_df_ablated["post_cleaned"].apply(
    lambda x: remove_terms(x, ablation_terms_ft)
)

test_df_ablated = test_df.copy()
test_df_ablated["post_cleaned"] = test_df_ablated["post_cleaned"].apply(
    lambda x: remove_terms(x, ablation_terms_ft)
)

# Write ablated files
write_fasttext_file(train_df_ablated, "post_cleaned", "political_leaning", "ft_train_ablated.txt")
write_fasttext_file(test_df_ablated, "post_cleaned", "political_leaning", "ft_test_ablated.txt")

# Train ablated model
ft_model_ablated = fasttext.train_supervised(
    input="ft_train_ablated.txt",
    lr=best_ft_params["lr"],
    epoch=best_ft_params["epoch"],
    wordNgrams=best_ft_params["wordNgrams"],
    dim=50,
    minCount=1,
    loss="softmax",
    verbose=0
)

# Evaluate
y_pred_ablated = predict_fasttext_labels(ft_model_ablated, test_df_ablated["post_cleaned"].tolist())
_, _, _, ablated_f1 = evaluate_predictions(y_test_true_ft, y_pred_ablated)

print("Original F1:", round(ft_optuna_f1, 4))
print("Ablated F1:", round(ablated_f1, 4))
print("Change:", round(ablated_f1 - ft_optuna_f1, 4))

RuntimeError: Encountered NaN.

In [39]:
ft_model_ablated = fasttext.train_supervised(
    input="ft_train_ablated.txt",
    lr=best_ft_params["lr"],
    epoch=best_ft_params["epoch"],
    wordNgrams=1,
    dim=50,
    minCount=0,
    loss="softmax",
    verbose=0
)

In [40]:
y_pred_ablated = predict_fasttext_labels(ft_model_ablated, test_df_ablated["post_cleaned"].tolist())
_, _, _, ablated_f1 = evaluate_predictions(y_test_true_ft, y_pred_ablated)

print("Original F1:", round(ft_optuna_f1, 4))
print("Ablated F1:", round(ablated_f1, 4))
print("Change:", round(ablated_f1 - ft_optuna_f1, 4))

Original F1: 0.636
Ablated F1: 0.6233
Change: -0.0128


In [41]:
# Top 500 features ablation
shap_top_500_ft = shap_agg_ft.sort_values("mean_abs_shap", ascending=False).head(500)
lime_top_500_ft = lime_global_ft.head(500)

combined_500_ft = set(shap_top_500_ft["token"]).union(set(lime_top_500_ft["feature"]))
combined_500_ft_df = pd.DataFrame({"feature": sorted(combined_500_ft)})
combined_500_ft_df["lexicon_overlap"] = combined_500_ft_df["feature"].apply(
    lambda x: feature_overlaps_lexicon(x, political_lexicon)
)
ablation_terms_500_ft = combined_500_ft_df[combined_500_ft_df["lexicon_overlap"]]["feature"].tolist()
print("Top 500 ablation terms:", len(ablation_terms_500_ft))

Top 500 ablation terms: 181


In [42]:
train_df_ablated_500 = train_df.copy()
train_df_ablated_500["post_cleaned"] = train_df_ablated_500["post_cleaned"].apply(
    lambda x: remove_terms(x, ablation_terms_500_ft)
)

test_df_ablated_500 = test_df.copy()
test_df_ablated_500["post_cleaned"] = test_df_ablated_500["post_cleaned"].apply(
    lambda x: remove_terms(x, ablation_terms_500_ft)
)

write_fasttext_file(train_df_ablated_500, "post_cleaned", "political_leaning", "ft_train_ablated_500.txt")
write_fasttext_file(test_df_ablated_500, "post_cleaned", "political_leaning", "ft_test_ablated_500.txt")

ft_model_ablated_500 = fasttext.train_supervised(
    input="ft_train_ablated_500.txt",
    lr=best_ft_params["lr"],
    epoch=best_ft_params["epoch"],
    wordNgrams=1,
    dim=50,
    minCount=0,
    loss="softmax",
    verbose=0
)

y_pred_ablated_500 = predict_fasttext_labels(ft_model_ablated_500, test_df_ablated_500["post_cleaned"].tolist())
_, _, _, ablated_500_f1 = evaluate_predictions(y_test_true_ft, y_pred_ablated_500)

print("Original F1:", round(ft_optuna_f1, 4))
print("Ablated 500 F1:", round(ablated_500_f1, 4))
print("Change:", round(ablated_500_f1 - ft_optuna_f1, 4))

RuntimeError: Encountered NaN.

In [43]:
ft_model_ablated_500 = fasttext.train_supervised(
    input="ft_train_ablated_500.txt",
    lr=0.01,
    epoch=best_ft_params["epoch"],
    wordNgrams=1,
    dim=50,
    minCount=0,
    loss="softmax",
    verbose=0
)

In [44]:
y_pred_ablated_500 = predict_fasttext_labels(ft_model_ablated_500, test_df_ablated_500["post_cleaned"].tolist())
_, _, _, ablated_500_f1 = evaluate_predictions(y_test_true_ft, y_pred_ablated_500)

print("Original F1:", round(ft_optuna_f1, 4))
print("Ablated 500 F1:", round(ablated_500_f1, 4))
print("Change:", round(ablated_500_f1 - ft_optuna_f1, 4))

Original F1: 0.636
Ablated 500 F1: 0.3485
Change: -0.2876


In [45]:
ft_model_ablated_500 = fasttext.train_supervised(
    input="ft_train_ablated_500.txt",
    lr=0.05,
    epoch=best_ft_params["epoch"],
    wordNgrams=1,
    dim=50,
    minCount=0,
    loss="softmax",
    verbose=0
)

y_pred_ablated_500 = predict_fasttext_labels(ft_model_ablated_500, test_df_ablated_500["post_cleaned"].tolist())
_, _, _, ablated_500_f1 = evaluate_predictions(y_test_true_ft, y_pred_ablated_500)

print("Original F1:", round(ft_optuna_f1, 4))
print("Ablated 500 F1:", round(ablated_500_f1, 4))
print("Change:", round(ablated_500_f1 - ft_optuna_f1, 4))

Original F1: 0.636
Ablated 500 F1: 0.6114
Change: -0.0247


In [46]:
shap_top_1000_ft = shap_agg_ft.sort_values("mean_abs_shap", ascending=False).head(1000)
lime_top_1000_ft = lime_global_ft.head(1000)

combined_1000_ft = set(shap_top_1000_ft["token"]).union(set(lime_top_1000_ft["feature"]))
combined_1000_ft_df = pd.DataFrame({"feature": sorted(combined_1000_ft)})
combined_1000_ft_df["lexicon_overlap"] = combined_1000_ft_df["feature"].apply(
    lambda x: feature_overlaps_lexicon(x, political_lexicon)
)
ablation_terms_1000_ft = combined_1000_ft_df[combined_1000_ft_df["lexicon_overlap"]]["feature"].tolist()
print("Top 1000 ablation terms:", len(ablation_terms_1000_ft))

Top 1000 ablation terms: 331


In [47]:
train_df_ablated_1000 = train_df.copy()
train_df_ablated_1000["post_cleaned"] = train_df_ablated_1000["post_cleaned"].apply(
    lambda x: remove_terms(x, ablation_terms_1000_ft)
)

test_df_ablated_1000 = test_df.copy()
test_df_ablated_1000["post_cleaned"] = test_df_ablated_1000["post_cleaned"].apply(
    lambda x: remove_terms(x, ablation_terms_1000_ft)
)

write_fasttext_file(train_df_ablated_1000, "post_cleaned", "political_leaning", "ft_train_ablated_1000.txt")
write_fasttext_file(test_df_ablated_1000, "post_cleaned", "political_leaning", "ft_test_ablated_1000.txt")

ft_model_ablated_1000 = fasttext.train_supervised(
    input="ft_train_ablated_1000.txt",
    lr=0.05,
    epoch=best_ft_params["epoch"],
    wordNgrams=1,
    dim=50,
    minCount=0,
    loss="softmax",
    verbose=0
)

y_pred_ablated_1000 = predict_fasttext_labels(ft_model_ablated_1000, test_df_ablated_1000["post_cleaned"].tolist())
_, _, _, ablated_1000_f1 = evaluate_predictions(y_test_true_ft, y_pred_ablated_1000)

print("Original F1:", round(ft_optuna_f1, 4))
print("Ablated 1000 F1:", round(ablated_1000_f1, 4))
print("Change:", round(ablated_1000_f1 - ft_optuna_f1, 4))

Original F1: 0.636
Ablated 1000 F1: 0.6079
Change: -0.0281


In [48]:
ablation_final_ft = pd.DataFrame({
    "experiment": [
        "Original FastText",
        "Ablation 110 terms (top 200)",
        "Ablation 181 terms (top 500)",
        "Ablation 331 terms (top 1000)"
    ],
    "num_terms_removed": [0, 110, 181, 331],
    "macro_f1": [
        round(ft_optuna_f1, 4),
        round(ablated_f1, 4),
        round(ablated_500_f1, 4),
        round(ablated_1000_f1, 4)
    ],
    "f1_change": [
        0,
        round(ablated_f1 - ft_optuna_f1, 4),
        round(ablated_500_f1 - ft_optuna_f1, 4),
        round(ablated_1000_f1 - ft_optuna_f1, 4)
    ]
})

ablation_final_ft

,experiment,num_terms_removed,macro_f1,f1_change
0,Original FastText,0,0.6360,0.0000
1,Ablation 110 terms (top 200),110,0.6233,-0.0128
2,Ablation 181 terms (top 500),181,0.6114,-0.0247
3,Ablation 331 terms (top 1000),331,0.6079,-0.0281


In [49]:
ablation_comparison = pd.DataFrame({
    "experiment": [
        "Original",
        "Ablation top 200 features",
        "Ablation top 500 features",
        "Ablation top 1000 features"
    ],
    "fasttext_macro_f1": [
        round(ft_optuna_f1, 4),
        round(ablated_f1, 4),
        round(ablated_500_f1, 4),
        round(ablated_1000_f1, 4)
    ],
    "fasttext_f1_change": [
        0,
        round(ablated_f1 - ft_optuna_f1, 4),
        round(ablated_500_f1 - ft_optuna_f1, 4),
        round(ablated_1000_f1 - ft_optuna_f1, 4)
    ],
    "modernbert_macro_f1": [
        0.6278, 0.6197, 0.6224, 0.6258
    ],
    "modernbert_f1_change": [
        0, -0.0081, -0.0054, -0.002
    ]
})

ablation_comparison.to_csv("ablation_comparison_both_models.csv", index=False)
ablation_comparison

,experiment,fasttext_macro_f1,fasttext_f1_change,modernbert_macro_f1,modernbert_f1_change
0,Original,0.6360,0.0000,0.6278,0.0000
1,Ablation top 200 features,0.6233,-0.0128,0.6197,-0.0081
2,Ablation top 500 features,0.6114,-0.0247,0.6224,-0.0054
3,Ablation top 1000 features,0.6079,-0.0281,0.6258,-0.0020


In [50]:
overlap_comparison = pd.DataFrame({
    "source": ["SHAP", "LIME", "Combined"],
    "fasttext_overlap_ratio": [0.245, 0.305, 0.275],
    "modernbert_overlap_ratio": [0.265, 0.375, 0.319]
})

overlap_comparison.to_csv("overlap_comparison_both_models.csv", index=False)
overlap_comparison

,source,fasttext_overlap_ratio,modernbert_overlap_ratio
0,SHAP,0.245,0.265
1,LIME,0.305,0.375
2,Combined,0.275,0.319


In [51]:
print("FastText SHAP top 30:")
print(shap_top_200_ft[["token", "direction", "mean_abs_shap"]].head(30).to_string())

print("\nFastText LIME top 30:")
print(lime_top_200_ft[["feature", "direction", "mean_abs_lime"]].head(30).to_string())

FastText SHAP top 30:
             token direction  mean_abs_shap
4744     mentality      left       0.010380
283       angelica     right       0.007604
5213           nyt      left       0.006832
5506         pause     right       0.005964
5221    obligation     right       0.005692
6447        ripped     right       0.005655
1863     deceiving     right       0.005636
6372  restrictions     right       0.005600
1776          cups      left       0.005549
4496      longtime     right       0.005388
663       beholden     right       0.005367
5633          pits     right       0.005360
6256        remain     right       0.005013
5897   professions     right       0.004997
7874       tracked     right       0.004898
2059   dilapidated     right       0.004858
6763   sensitivity     right       0.004791
3929  institutions     right       0.004624
144       agencies     right       0.004580
7358        strife      left       0.004545
7628         tease      left       0.004415
2757      